# Toy Corpus Language Model in JAX

This notebook walks through a complete mini language-model workflow — from raw text to saved artifacts — rewritten from scratch in **JAX**.

## Why JAX instead of NumPy?

NumPy is wonderful for understanding shapes and array math, but modern deep learning has two requirements NumPy cannot satisfy:

1. **Automatic differentiation** — computing gradients by hand (as in the NumPy version) is tedious and error-prone for anything beyond a two-layer network. JAX provides `jax.grad` and `jax.value_and_grad`, which derive exact gradients from any pure Python/JAX function automatically.
2. **JIT compilation** — Python is slow. JAX's `jax.jit` traces your function once and compiles it to highly optimised XLA bytecode, giving you near-C performance without leaving Python.

Beyond those two headline features, JAX's *functional* design — no hidden state, no in-place mutation — maps almost perfectly onto how research papers describe models mathematically. That correspondence makes code easier to read, debug, and reason about.

## What you will build

| Step | What happens |
|------|--------------|
| 1 | Imports, PRNG setup, hyperparameters |
| 2 | Tiny in-memory corpus |
| 3 | Tokenisation and vocabulary |
| 4 | Sliding-window training examples |
| 5 | Model definition (embedding → pool → project) |
| 6 | Loss function with JAX autodiff |
| 7 | JIT-compiled training loop |
| 8 | Evaluation on sample prompts |
| 9 | Save and reload artifacts |

## How to read this notebook

- Read the markdown cell first — it explains the *why*.
- Then run the code cell below it.
- Compare output to what the commentary predicts.
- If something breaks, the markdown cell will tell you what to check first.

Try small experiments once the full notebook runs cleanly:
- Change `SEQ_LEN` and observe how context width affects predictions.
- Increase `EMBED_DIM` and watch whether loss improves or overfits.
- Add sentences to the corpus and see new words appear in top-k predictions.

---
## Cell 1 — Imports and Configuration

### JAX imports explained

```python
import jax
import jax.numpy as jnp
from jax import grad, jit, value_and_grad
import optax
```

- **`jax`** — the top-level package. Provides `jax.random`, `jax.grad`, `jax.jit`, `jax.vmap`, etc.
- **`jax.numpy`** — a near-complete reimplementation of NumPy's API. Almost every `np.foo` has a `jnp.foo` equivalent. The key difference: JAX arrays are *immutable* and live on an accelerator (CPU/GPU/TPU). You never write `arr[i] = x`; instead you use `arr.at[i].set(x)` which returns a new array.
- **`grad` / `value_and_grad`** — the autodiff workhorses. `grad(f)(x)` returns `∂f/∂x`. `value_and_grad(f)(x)` returns `(f(x), ∂f/∂x)` in one pass — useful because we usually want the loss number *and* its gradient simultaneously.
- **`jit`** — wraps a function so it is compiled on first call and cached for subsequent calls. The compilation is transparent: the function signature and behaviour do not change.
- **`optax`** — a gradient-processing library built on JAX. It provides optimisers (SGD, Adam, AdaGrad…) as pure transformations over a gradient tree, with no hidden state.

### PRNG in JAX

JAX handles randomness differently from NumPy. NumPy maintains a *global* random state that mutates when you call `np.random.randn`. This makes reproducibility hard in multi-device or parallelised settings.

JAX uses **explicit PRNG keys**. A key is just a pair of uint32 integers. Every random operation takes a key as input and is *deterministic* given that key. To get a fresh key, you split the current one:

```python
key = jax.random.PRNGKey(42)          # Seed the root key
key, subkey = jax.random.split(key)   # Fork: use subkey for this op, keep key for later
x = jax.random.normal(subkey, (3,))  # Deterministic given subkey
```

This pattern threads randomness explicitly through your code — verbose at first, but it eliminates an entire class of hard-to-find reproducibility bugs.

### Hyperparameters

| Name | Value | Controls |
|------|-------|----------|
| `SEQ_LEN` | 4 | Context window: how many preceding tokens the model sees |
| `BATCH_SIZE` | 8 | Examples per gradient step |
| `LEARNING_RATE` | 0.05 | SGD step size |
| `EPOCHS` | 300 | Full passes over training data |
| `EMBED_DIM` | 16 | Dimensionality of each word vector |

In [ ]:
import json
import pickle
from pathlib import Path

import numpy as np           # Used only for data prep; model math lives in JAX
import jax
import jax.numpy as jnp
from jax import jit, value_and_grad
import optax

# ── PRNG root key ──────────────────────────────────────────────────────────────
# We derive all random keys from this single root. Every time we need randomness
# we split off a subkey. The root key advances so no two operations share a key.
ROOT_KEY = jax.random.PRNGKey(42)

# ── Hyperparameters ────────────────────────────────────────────────────────────
SEQ_LEN       = 4
BATCH_SIZE    = 8
LEARNING_RATE = 0.05
EPOCHS        = 300
EMBED_DIM     = 16

ARTIFACT_DIR = Path("artifacts")
ARTIFACT_DIR.mkdir(exist_ok=True)

print(f"JAX version : {jax.__version__}")
print(f"Default backend : {jax.default_backend()}")
print(f"SEQ_LEN={SEQ_LEN}, BATCH_SIZE={BATCH_SIZE}, "
      f"LR={LEARNING_RATE}, EPOCHS={EPOCHS}, EMBED_DIM={EMBED_DIM}")

---
## Cell 2 — Build the Toy Corpus

We create ten sentences directly in memory. A larger or disk-based corpus would work identically — only the `corpus` list would change.

**Why a tiny corpus?**
- Training takes under a second, so you can iterate immediately.
- The vocabulary is small enough to inspect manually.
- Repeated words and phrases give the model a measurable signal to learn.

**What to notice in the output:**
- Ten sentences — one per list entry.
- Varied structure, but many shared words (`the`, `we`, `a`, `model`).
- Consistent lowercase — important because our tokeniser is case-sensitive.

In [ ]:
corpus = [
    "the model learns from text",
    "the tokenizer splits text into pieces",
    "byte pair encoding merges common pairs",
    "small corpora are good for debugging",
    "we train a tiny language model",
    "the residual stream tracks progress",
    "attention starts with token representations",
    "we test before we scale",
    "paper notes guide implementation choices",
    "weekly builds record what broke",
]

print(f"Number of sentences : {len(corpus)}")
print()
for i, line in enumerate(corpus, start=1):
    print(f"  {i:2d}. {line}")

---
## Cell 3 — Tokenise and Encode the Corpus

### The tokenisation pipeline

Neural networks cannot operate on strings. Every word must be mapped to an integer, and every integer to a dense vector. That two-step process — `word → id → vector` — is the foundation of all language modelling.

Our simple word-level tokeniser:

```
raw text  →  lowercase + split on spaces  →  token list
token list  →  sorted unique set  →  vocabulary
vocabulary  →  enumerate  →  word_to_id dict
token list  →  word_to_id lookup  →  encoded integer sequence
```

Real-world tokenisers (BPE, SentencePiece, WordPiece) are more sophisticated — they split rare words into subwords so the vocabulary stays manageable — but the integer-mapping step is identical.

### What to verify

- **Vocabulary size** should be around 40–50 for these ten sentences.
- **Token ↔ ID alignment**: the printed token and its integer should correspond — no off-by-one shifts.
- **Encoded stream**: a long sequence of integers, one per word in the whole corpus.

In [ ]:
# ── Tokenise ───────────────────────────────────────────────────────────────────
def tokenize(lines):
    """Lowercase-split each line and flatten into one token list."""
    tokens = []
    for line in lines:
        tokens.extend(line.lower().split())
    return tokens

all_tokens = tokenize(corpus)

# ── Build vocabulary ───────────────────────────────────────────────────────────
# Sorting is important: it makes the mapping deterministic across Python runs.
vocab = sorted(set(all_tokens))
word_to_id = {w: i for i, w in enumerate(vocab)}
id_to_word = {i: w for w, i in word_to_id.items()}
VOCAB_SIZE = len(vocab)

# ── Encode ─────────────────────────────────────────────────────────────────────
# NumPy array here; we'll convert to JAX arrays inside the training loop.
encoded = np.array([word_to_id[w] for w in all_tokens], dtype=np.int32)

print(f"Vocabulary size  : {VOCAB_SIZE}")
print(f"Total tokens     : {len(all_tokens)}")
print()
print("First 14 tokens  :", all_tokens[:14])
print("First 14 IDs     :", encoded[:14].tolist())
print()
print("Vocabulary (first 20):", vocab[:20])

---
## Cell 4 — Build Sliding-Window Training Examples

### The next-token prediction objective

A language model is trained to answer one question repeatedly:

> Given the last *k* tokens, what is the most likely next token?

We formalise this as a supervised learning problem by sliding a window of width `SEQ_LEN` across the encoded stream:

```
encoded : [5, 22, 18, 7, 40, 11, ...]

window 0:  X = [5,  22, 18,  7]   y = 40
window 1:  X = [22, 18,  7, 40]   y = 11
window 2:  X = [18,  7, 40, 11]   y = ...
```

This produces `len(encoded) - SEQ_LEN` training pairs.

### Why we use plain NumPy here

Data preparation — slicing, shuffling, batching — does not need to be differentiated, so plain NumPy is fine. We convert to JAX arrays inside the training loop, right before the forward pass. This keeps data wrangling separate from numerical computation, which makes both easier to debug.

### Shape contract

| Array | Shape | Dtype |
|-------|-------|-------|
| `X` | `(num_examples, SEQ_LEN)` | int32 |
| `y` | `(num_examples,)` | int32 |

Violating these shapes will surface as cryptic matmul errors later. Check them now.

In [ ]:
def build_examples(token_ids, seq_len):
    """Sliding-window next-token pairs."""
    X, y = [], []
    for i in range(len(token_ids) - seq_len):
        X.append(token_ids[i : i + seq_len])
        y.append(token_ids[i + seq_len])
    return np.array(X, dtype=np.int32), np.array(y, dtype=np.int32)

X_np, y_np = build_examples(encoded, SEQ_LEN)

print(f"X shape : {X_np.shape}   (num_examples × SEQ_LEN)")
print(f"y shape : {y_np.shape}   (num_examples,)")
print()
# Show the first three examples in human-readable form
for i in range(3):
    ctx   = " ".join(id_to_word[t] for t in X_np[i])
    tgt   = id_to_word[y_np[i]]
    print(f"  Example {i}: [{ctx}]  →  '{tgt}'")

---
## Cell 5 — Define the Model

### Model architecture

Our model has three components, applied in sequence:

```
Input:  batch of integer sequences,  shape (B, SEQ_LEN)
   │
   ▼ Embedding lookup
   │  Each integer → EMBED_DIM-dimensional vector
   │  Shape: (B, SEQ_LEN, EMBED_DIM)
   │
   ▼ Mean pooling over sequence positions
   │  Average all SEQ_LEN vectors into one context vector
   │  Shape: (B, EMBED_DIM)
   │
   ▼ Linear projection
   │  Matrix W + bias b maps context vector → vocabulary logits
   │  Shape: (B, VOCAB_SIZE)
   │
   ▼ Softmax (for evaluation only — loss uses log-softmax internally)
      Shape: (B, VOCAB_SIZE)  — probabilities summing to 1
```

### Parameters as a plain Python dict

JAX does not have a built-in `Module` class. Parameters are just **nested Python dicts (or lists) of JAX arrays** — sometimes called a *pytree*. JAX's `grad` and `jit` understand pytrees natively, so you can compute gradients with respect to an entire dict in one call.

```python
params = {
    "E" : jnp.array(...)   # shape (VOCAB_SIZE, EMBED_DIM)
    "W" : jnp.array(...)   # shape (EMBED_DIM, VOCAB_SIZE)
    "b" : jnp.array(...)   # shape (VOCAB_SIZE,)
}
```

This explicitness is intentional. When something goes wrong you can print any parameter, check its shape, inspect its gradient, and reason about it without digging through framework internals.

### Why mean pooling?

Mean pooling is the simplest way to aggregate a variable-length sequence into a fixed-size vector. It has no learned parameters and treats all positions equally. Real models use attention — a *selective* weighted average — but mean pooling is a fine stand-in when the goal is understanding the pipeline rather than maximising accuracy.

### Initialisation

We use small Gaussian noise (`stddev=0.01`) so the softmax starts close to uniform. If you initialise with large values, early gradients explode and training never recovers cleanly.

In [ ]:
# ── Parameter initialisation ───────────────────────────────────────────────────
ROOT_KEY, k1, k2 = jax.random.split(ROOT_KEY, 3)

params = {
    "E": jax.random.normal(k1, (VOCAB_SIZE, EMBED_DIM)) * 0.01,   # Embedding table
    "W": jax.random.normal(k2, (EMBED_DIM, VOCAB_SIZE)) * 0.01,   # Output projection
    "b": jnp.zeros(VOCAB_SIZE),                                    # Output bias
}

# ── Forward pass ───────────────────────────────────────────────────────────────
def forward(params, batch_X):
    """
    batch_X : int32 array of shape (B, SEQ_LEN)
    returns  : float32 logits of shape (B, VOCAB_SIZE)
    """
    # Step 1 — Embedding lookup
    # params["E"][batch_X] gathers rows from E for each token ID.
    # Result shape: (B, SEQ_LEN, EMBED_DIM)
    embedded = params["E"][batch_X]

    # Step 2 — Mean pool over the sequence dimension (axis=1)
    # This collapses (B, SEQ_LEN, EMBED_DIM) → (B, EMBED_DIM)
    hidden = embedded.mean(axis=1)

    # Step 3 — Linear projection to vocabulary logits
    # (B, EMBED_DIM) @ (EMBED_DIM, VOCAB_SIZE) + (VOCAB_SIZE,) → (B, VOCAB_SIZE)
    logits = hidden @ params["W"] + params["b"]

    return logits   # raw logits; softmax applied at eval time


# ── Quick shape sanity check ───────────────────────────────────────────────────
dummy_X = jnp.array(X_np[:4], dtype=jnp.int32)
dummy_logits = forward(params, dummy_X)

print("Parameter shapes:")
for k, v in params.items():
    print(f"  params['{k}'] : {v.shape}")
print()
print(f"forward() output shape : {dummy_logits.shape}")
print("  (batch_size × VOCAB_SIZE — one logit per vocabulary word per example)")

---
## Cell 6 — Loss Function and JAX Autodiff

### Cross-entropy loss

Cross-entropy measures how surprised the model is by the true next token. Given:
- `logits` — raw scores for every vocabulary word (before softmax)
- `targets` — integer indices of the true next tokens

The loss for a single example is:

```
loss = -log( softmax(logits)[target] )
     = -log( exp(logit[target]) / Σ exp(logit[j]) )
     = -logit[target] + log( Σ exp(logit[j]) )
```

We use `jax.nn.log_softmax` rather than computing softmax then taking the log. This combined operation is numerically stabler: the `log(exp(...))` chain avoids catastrophic cancellation from very large or very small exponentials.

### `value_and_grad` — the workhorse of JAX training

```python
loss_val, grads = value_and_grad(loss_fn)(params, X_batch, y_batch)
```

`value_and_grad(f)` returns a new function that, when called, gives back both the function output *and* the gradient of that output with respect to the first argument (`params`). The gradient has **exactly the same structure** as `params` — a dict of JAX arrays with identical shapes. This is the pytree property: JAX preserves data structure through automatic differentiation.

### Optax optimiser

Optax transforms gradients through a *chain* of operations before applying them. Plain SGD is the simplest chain:

```
gradient  →  scale by -learning_rate  →  add to params
```

Optax stores any internal state (e.g. momentum buffers) in an `opt_state` pytree that you carry explicitly. This keeps the optimiser stateless from the outside, which is important for JIT-compiling the update step.

In [ ]:
# ── Loss function ──────────────────────────────────────────────────────────────
def cross_entropy_loss(params, batch_X, batch_y):
    """
    Compute mean cross-entropy loss over a batch.

    batch_X : int32  (B, SEQ_LEN)
    batch_y : int32  (B,)
    returns : scalar float32
    """
    logits    = forward(params, batch_X)               # (B, VOCAB_SIZE)
    log_probs = jax.nn.log_softmax(logits, axis=-1)    # numerically stable log-softmax

    # Gather the log-probability of the correct next token for each example
    # jnp.arange(B) provides row indices; batch_y provides column indices.
    correct_log_probs = log_probs[jnp.arange(len(batch_y)), batch_y]  # (B,)

    return -correct_log_probs.mean()   # scalar — lower is better


# ── Optimiser ─────────────────────────────────────────────────────────────────
# optax.sgd is a simple wrapper: grads are scaled by -lr and added to params.
optimizer  = optax.sgd(LEARNING_RATE)
opt_state  = optimizer.init(params)

# ── Verify loss is computable and gradient has the right structure ─────────────
dummy_X_jax = jnp.array(X_np[:8], dtype=jnp.int32)
dummy_y_jax = jnp.array(y_np[:8], dtype=jnp.int32)

init_loss, init_grads = value_and_grad(cross_entropy_loss)(params, dummy_X_jax, dummy_y_jax)

print(f"Initial loss (random params) : {init_loss:.4f}")
print(f"  Expected near log(VOCAB_SIZE) = {np.log(VOCAB_SIZE):.4f}  (uniform baseline)")
print()
print("Gradient shapes (should match params):")
for k, g in init_grads.items():
    print(f"  grads['{k}'] : {g.shape}")

---
## Cell 7 — JIT-Compiled Training Loop

### What `@jit` does

Decorating `train_step` with `@jit` tells JAX to *trace* the function on first call — recording every operation as a computation graph — and then compile that graph to XLA. Subsequent calls skip Python entirely and run the compiled kernel directly.

For this tiny model the speedup is modest, but on real models `@jit` is the difference between training being practical and impractical.

**Constraints introduced by JIT:**
1. **No Python side-effects** — you cannot print or mutate global variables inside a JIT-compiled function. Anything that should propagate must be returned explicitly.
2. **Array shapes must be static** — JAX recompiles if shapes change. Pad your batches to a fixed size if needed.
3. **No Python control flow on traced values** — `if jnp_array > 0` is not allowed; use `jnp.where` instead.

### The update step

Every training step follows the same four-line pattern:

```python
loss, grads    = value_and_grad(loss_fn)(params, X, y)  # 1. Forward + backward
updates, state = optimizer.update(grads, opt_state)     # 2. Transform gradients
new_params     = optax.apply_updates(params, updates)   # 3. Apply updates
return new_params, new_state, loss                      # 4. Return new state
```

Notice that `params` and `opt_state` are *returned*, not mutated. JAX functional style: everything is an input→output transformation.

### Mini-batch shuffling

We shuffle the training data at the start of each epoch using a JAX random permutation. This prevents the model from memorising the order of examples (a common failure mode in small datasets).

### Expected behaviour

- Loss should trend downward from ~3.8 (log VOCAB_SIZE ≈ log 47 ≈ 3.85) toward 1–2.
- Some per-epoch fluctuation is normal — mini-batches introduce noise.
- If loss is flat or rising, check learning rate and gradient signs.

In [ ]:
# ── JIT-compiled parameter update ─────────────────────────────────────────────
@jit
def train_step(params, opt_state, batch_X, batch_y):
    """
    One gradient-descent step.
    All arguments are JAX arrays or pytrees — safe to JIT.
    Returns updated params, updated optimizer state, and the scalar loss.
    """
    loss, grads   = value_and_grad(cross_entropy_loss)(params, batch_X, batch_y)
    updates, new_opt_state = optimizer.update(grads, opt_state)
    new_params    = optax.apply_updates(params, updates)
    return new_params, new_opt_state, loss


# ── Training loop ─────────────────────────────────────────────────────────────
loss_history = []
N = len(X_np)

for epoch in range(1, EPOCHS + 1):
    # Shuffle data with a fresh JAX key each epoch
    ROOT_KEY, shuf_key = jax.random.split(ROOT_KEY)
    perm  = jax.random.permutation(shuf_key, N)
    X_shuf = X_np[np.array(perm)]   # NumPy indexing for data prep
    y_shuf = y_np[np.array(perm)]

    epoch_losses = []

    for start in range(0, N, BATCH_SIZE):
        end      = min(start + BATCH_SIZE, N)
        batch_X  = jnp.array(X_shuf[start:end], dtype=jnp.int32)
        batch_y  = jnp.array(y_shuf[start:end], dtype=jnp.int32)

        params, opt_state, loss = train_step(params, opt_state, batch_X, batch_y)
        epoch_losses.append(float(loss))

    avg_loss = float(np.mean(epoch_losses))
    loss_history.append(avg_loss)

    if epoch == 1 or epoch % 50 == 0:
        print(f"Epoch {epoch:4d} | loss = {avg_loss:.4f}")

print()
print("Training complete.")
print(f"Loss: {loss_history[0]:.4f}  →  {loss_history[-1]:.4f}")

---
## Cell 8 — Visualise the Loss Curve

A training curve is the fastest diagnostic tool you have. Before inspecting predictions, check:

- **Does the curve trend downward overall?** If not, learning rate is wrong or data preparation is broken.
- **Is there a sharp early drop followed by a plateau?** Common for small models — they saturate quickly.
- **Is the curve noisy/spiky?** Expected with mini-batches. Smooth it by increasing `BATCH_SIZE` or decreasing `LEARNING_RATE`.
- **Does the curve diverge (shoot up)?** Learning rate is too high.

In [ ]:
try:
    import matplotlib.pyplot as plt

    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(range(1, EPOCHS + 1), loss_history, linewidth=1.5, color="steelblue")
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Cross-entropy loss")
    ax.set_title("Training loss curve")
    ax.axhline(np.log(VOCAB_SIZE), color="salmon", linestyle="--",
               label=f"Uniform baseline  log({VOCAB_SIZE}) ≈ {np.log(VOCAB_SIZE):.2f}")
    ax.legend()
    plt.tight_layout()
    plt.savefig(ARTIFACT_DIR / "loss_curve.png", dpi=120)
    plt.show()
    print("Loss curve saved to artifacts/loss_curve.png")
except ImportError:
    print("matplotlib not available — printing loss summary instead.")
    for epoch in [1, 50, 100, 150, 200, 250, 300]:
        if epoch <= len(loss_history):
            print(f"  Epoch {epoch:4d} | {loss_history[epoch-1]:.4f}")

---
## Cell 9 — Evaluate on Sample Prompts

### What we're checking

After training we probe the model with short prompts and print the **top-k most likely next words**.

With only ten training sentences the model will not produce grammatically perfect continuations. But it should show *learned associations*:

- Prompts ending in `the` → model should favour nouns that follow `the` in the corpus.
- Prompts ending in `we` → model should favour `train`, `test`, `scale`.
- Prompts from unseen contexts → predictions will be noisier.

### Padding

If a prompt is shorter than `SEQ_LEN`, we left-pad with `vocab[0]` (the first alphabetically sorted word). This is a common practical convention. Alternative: use a dedicated `<pad>` token. The key requirement is that the input to the model always has the expected shape.

### Softmax at inference time

During training we use `log_softmax` inside the loss function. At inference time we want *probabilities* to display to users, so we apply a regular `jax.nn.softmax` to the raw logits.

In [ ]:
def predict_next(prompt_words, params, top_k=5):
    """
    Given a list of word strings, predict the top-k most likely next words.

    Steps:
      1. Lowercase all input words.
      2. Left-pad or right-truncate to SEQ_LEN.
      3. Map words to IDs (unknown words map to 0).
      4. Run forward pass and apply softmax.
      5. Return sorted (word, probability) pairs.
    """
    words = [w.lower() for w in prompt_words]

    # Pad or truncate to exactly SEQ_LEN
    if len(words) < SEQ_LEN:
        words = [vocab[0]] * (SEQ_LEN - len(words)) + words
    else:
        words = words[-SEQ_LEN:]

    ids    = jnp.array([[word_to_id.get(w, 0) for w in words]], dtype=jnp.int32)
    logits = forward(params, ids)                        # (1, VOCAB_SIZE)
    probs  = jax.nn.softmax(logits[0])                  # (VOCAB_SIZE,)

    top_ids = jnp.argsort(probs)[-top_k:][::-1]         # Descending by probability
    return [(id_to_word[int(i)], float(probs[i])) for i in top_ids]


# ── Run sample prompts ─────────────────────────────────────────────────────────
prompts = [
    ["the", "tokenizer", "splits", "text"],
    ["we", "test", "before", "we"],
    ["paper", "notes", "guide", "implementation"],
    ["the", "model", "learns", "from"],
    ["byte", "pair", "encoding", "merges"],
]

for prompt in prompts:
    preds = predict_next(prompt, params, top_k=5)
    ctx   = ' '.join(prompt)
    print(f"Prompt : '{ctx}'")
    print("Top 5  :", [(w, f"{p:.3f}") for w, p in preds])
    print()

---
## Cell 10 — Sanity Checks

Automated assertions protect against silent regressions. These three checks verify:

1. **Training ran** — `loss_history` has the right length.
2. **Training helped** — final loss is lower than initial loss.
3. **Predictions are well-formed** — `predict_next` returns the requested number of candidates.

Run this cell every time you change a hyperparameter. If any assertion fails, the error message tells you exactly what to investigate.

In [ ]:
# Assertion 1: training loop ran the correct number of epochs
assert len(loss_history) == EPOCHS, (
    f"Expected {EPOCHS} loss entries, got {len(loss_history)}"
)

# Assertion 2: loss decreased — the model actually learned something
assert loss_history[-1] < loss_history[0], (
    f"Loss did not decrease: {loss_history[0]:.4f} → {loss_history[-1]:.4f}. "
    f"Check learning rate ({LEARNING_RATE}) and data shapes."
)

# Assertion 3: prediction output has the right structure
test_top = predict_next(["the", "model", "learns", "from"], params, top_k=3)
assert len(test_top) == 3, f"Expected 3 predictions, got {len(test_top)}"
assert all(isinstance(w, str) and 0 < p < 1 for w, p in test_top), (
    "Predictions should be (str, float in (0,1)) pairs"
)

# Assertion 4: probabilities are a valid distribution (sum to ~1)
dummy_logits = forward(params, jnp.array(X_np[:1], dtype=jnp.int32))
prob_sum = float(jax.nn.softmax(dummy_logits[0]).sum())
assert abs(prob_sum - 1.0) < 1e-5, f"Probabilities sum to {prob_sum}, not 1"

print("All sanity checks passed ✓")
print(f"  Epochs run       : {len(loss_history)}")
print(f"  Loss improvement : {loss_history[0]:.4f} → {loss_history[-1]:.4f}")
print(f"  Sample preds     : {test_top}")
print(f"  Prob sum check   : {prob_sum:.8f}")

---
## Cell 11 — Save and Reload Artifacts

### What we save and why

A workflow is not complete until you can persist and restore every artifact needed for inference.

| File | Format | Contains |
|------|--------|----------|
| `toy_vocab.json` | JSON | `word_to_id` and `id_to_word` dicts |
| `toy_model_params.pkl` | Pickle | Trained JAX parameter pytree |
| `loss_curve.png` | PNG | Training loss curve |

**Why JSON for vocab?** JSON is human-readable, language-agnostic, and stable across Python versions. You can open it in any text editor, grep for a word, or load it in a JavaScript or Go service.

**Why Pickle for params?** JAX arrays serialise cleanly through pickle. For production use, prefer `orbax` (the JAX-native checkpoint library) or `numpy.save` with `.npy` files — pickle is not cross-language and has security caveats with untrusted data. For a toy notebook, it's fine.

### The reload verification pattern

After every save, reload and verify. Specifically:

1. Check that file sizes are non-zero (saves can silently fail on full disks).
2. Check that reloaded vocab size matches original.
3. Run a forward pass with reloaded params and confirm output is identical to the pre-save run.

Step 3 is the strongest check: it validates the entire serialisation round-trip end-to-end.

In [ ]:
vocab_path  = ARTIFACT_DIR / "toy_vocab.json"
params_path = ARTIFACT_DIR / "toy_model_params.pkl"

# ── Save ──────────────────────────────────────────────────────────────────────
with open(vocab_path, "w", encoding="utf-8") as f:
    # JSON requires string keys; id_to_word keys are ints so convert them
    json.dump(
        {"word_to_id": word_to_id, "id_to_word": {str(k): v for k, v in id_to_word.items()}},
        f,
        indent=2,
    )

# Convert JAX arrays to NumPy before pickling for maximum compatibility
params_np = {k: np.array(v) for k, v in params.items()}
with open(params_path, "wb") as f:
    pickle.dump(params_np, f)

print(f"Saved vocab  : {vocab_path}   ({vocab_path.stat().st_size} bytes)")
print(f"Saved params : {params_path}  ({params_path.stat().st_size} bytes)")

# ── Reload ────────────────────────────────────────────────────────────────────
with open(vocab_path, "r", encoding="utf-8") as f:
    loaded_vocab = json.load(f)

with open(params_path, "rb") as f:
    loaded_params_np = pickle.load(f)

# Convert back to JAX arrays
loaded_params = {k: jnp.array(v) for k, v in loaded_params_np.items()}

# ── Verify ────────────────────────────────────────────────────────────────────
assert len(loaded_vocab["word_to_id"]) == VOCAB_SIZE
assert set(loaded_params.keys()) == set(params.keys())

# Forward-pass equivalence check
probe   = jnp.array(X_np[:4], dtype=jnp.int32)
orig    = np.array(forward(params,        probe))
reloaded= np.array(forward(loaded_params, probe))
assert np.allclose(orig, reloaded, atol=1e-6), "Reloaded params produce different outputs!"

print()
print("Reload and equivalence check passed ✓")
print(f"  Vocab size (original / reloaded) : {VOCAB_SIZE} / {len(loaded_vocab['word_to_id'])}")
print(f"  Param keys : {list(loaded_params.keys())}")
print(f"  Max abs diff in logits : {np.abs(orig - reloaded).max():.2e}")

---
## Cell 12 — Recap and Next Steps

You have now built a complete language-modelling pipeline in JAX:

```
raw text
  └─ tokenise → vocabulary → integer encoding
       └─ sliding window → (X, y) training pairs
            └─ param dict (E, W, b)  ←  random init
                 └─ forward()  →  logits
                      └─ cross_entropy_loss()  →  scalar
                           └─ value_and_grad()  →  gradients
                                └─ optax.sgd update  →  new params
                                     └─ save to disk  →  vocab.json + params.pkl
```

### JAX concepts used

| Concept | Where used | Why it matters |
|---------|------------|----------------|
| Explicit PRNG keys | Init, shuffle | Reproducibility without global state |
| `jnp` arrays (immutable) | Everywhere | Safe to JIT and differentiate |
| `value_and_grad` | `train_step` | Forward + backward in one call |
| `@jit` | `train_step` | XLA compilation for speed |
| Pytrees (dict of arrays) | `params` | Gradient structure mirrors param structure |
| `optax` optimiser | Training loop | Stateless gradient transformation |

### Suggested experiments

1. **Swap SGD for Adam**: change `optax.sgd(LR)` to `optax.adam(LR)` and compare loss curves.
2. **Add a hidden layer**: between mean pooling and the output projection, add `hidden2 = jax.nn.relu(hidden @ W1 + b1)`.
3. **Replace mean pooling with learned attention**: implement a single-head dot-product attention over the SEQ_LEN positions.
4. **Try a larger corpus**: load a text file line-by-line instead of the hardcoded list.
5. **Use `jax.vmap`**: vectorise `predict_next` over a batch of prompts instead of looping in Python.